To run this notebook, you will need a `.env` file at the root of the project. It should contain the following keys:
```
LLM_SERVICE=Azure
AZURE_OPENAI_API_KEY=...
AZURE_OPENAI_ENDPOINT=...
AZURE_OPENAI_DEPLOYMENT_NAME=...
AZURE_OPENAI_API_VERSION=...
```

The first step is to load the libraries we'll be using. I am importing `pandas` for basic data manipulation, and a package called [discovery_utils](https://github.com/nestauk/discovery_utils) that Karlis made. It contains various functions that we have used and reused across Discovery projects, including functions for extracting structured information using LLMs. You can read more about how the `llm` module works [here](https://github.com/nestauk/discovery_utils/wiki/Checking-data-with-LLM).

In [ ]:
import pandas as pd

from discovery_utils.utils.llm import batch_check

from discovery_heat_pump_futures import PROJECT_DIR

Karlis has created two datasets for this project: one containing research abstracts from [OpenAlex](https://openalex.org/), and one from [Google Patents](https://patents.google.com/).

In [ ]:
openalex_df = pd.read_csv("https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_openalex.csv")
patents_df = pd.read_json("https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_patents.json", lines=True)

The patents dataset looks like this:

In [ ]:
patents_df.head()

Below, we specify a helper function that can also be used on the OpenAlex data to concatenate the titles and abstracts of individual patents/abstracts. This new `"title_abstract"` field will be the input to `LLMProcessor`.

In [ ]:
def format_title_abstract(df: pd.DataFrame, title: str='title', abstract: str='abstract') -> pd.DataFrame:
    """
    Format the title and abstract for LLM input.
    """
    df['title_abstract'] = (
    'TITLE: ' + df['title'].str.lower().fillna('') +
    ' ABSTRACT: ' + df['abstract'].str.lower().fillna('')
    )
    return df

patents_df = format_title_abstract(patents_df, title='title', abstract='abstract')
patents_df.head()

We need to define a system message to tell the LLM what it should do, and define exactly what outputs we want to get back, and what format they should be in.

In [ ]:
system_message = "Determine whether this text presents an improvement to heat pump components or systems."

fields = [
    {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    {"name": "summary", "type": "str", "description": "Summarise the patent in plain English."},
    {"name": "components", "type": "List[str]", "description": "In terms of heat pump components, does this application mention refrigerants, the mechanical compressor, or heat exchange? Return all that apply."},
]

We will create a small sample of data and convert it into the correct format for passing to the LLM:

In [ ]:
patents_sample = patents_df.sample(5, random_state=42)

test_data = patents_sample[['url', 'title_abstract']].to_dict(orient='records')

test_data_dict = {str(i): text for i, text in enumerate(test_data)}

Now that we have test data, a system message and some defined output fields, we are ready to run `LLMProcessor`.

In [ ]:
outpath = PROJECT_DIR / "outputs/llm_check_output.jsonl"

processor = batch_check.LLMProcessor(
            model_name="gpt-4o-mini",
            temperature=0,
            output_path=str(outpath),
            system_message=system_message,
            session_name="test",
            output_fields=fields,
        )

task = processor.run(test_data_dict, batch_size=1, sleep_time=0.5)
await task

We can now read the output back in to see what we got!

In [ ]:
test_output = pd.read_json(outpath, lines=True)
test_output.head()